# Pipeline des ventes pour PostgreSQL

Ce notebook construit un pipeline Pandas complet pour `orders.csv`, `people.csv` et `returns.csv`.
L'objectif est de produire un CSV final propre et a plat, pret a etre importe plus tard dans PostgreSQL via PgAdmin.

## 1. Configuration

Cette etape prepare l'environnement de travail avec la bibliotheque standard Python et Pandas uniquement.

In [ ]:
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

BASE_DIR = Path.cwd()
ORDERS_PATH = BASE_DIR / "orders.csv"
PEOPLE_PATH = BASE_DIR / "people.csv"
RETURNS_PATH = BASE_DIR / "returns.csv"
OUTPUT_PATH = BASE_DIR / "sales_final_clean.csv"

print(f"Working directory: {BASE_DIR}")
print(f"Output file: {OUTPUT_PATH.name}")

## 2. Chargement des fichiers CSV

Les noms de colonnes sont normalises en `snake_case` des le depart pour rendre le pipeline plus lisible et compatible avec SQL.

In [ ]:
def to_snake_case(name: str) -> str:
    name = name.strip().lower()
    name = name.replace("-", "_").replace("/", "_")
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return re.sub(r"_+", "_", name).strip("_")


def normalize_text_value(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).replace("\xa0", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text if text else pd.NA


def normalize_text_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    text_columns = df.select_dtypes(include=["object", "string"]).columns
    for column in text_columns:
        df[column] = df[column].map(normalize_text_value)
    return df


def clean_postal_code(series: pd.Series) -> pd.Series:
    series = series.astype("string").str.strip()
    series = series.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    series = series.str.replace(r"\.0$", "", regex=True)
    return series


def build_quality_summary(name: str, df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "dataset": name,
            "rows": len(df),
            "columns": len(df.columns),
            "duplicate_rows": int(df.duplicated().sum()),
            "missing_cells": int(df.isna().sum().sum()),
        }
    ])


orders_raw = pd.read_csv(ORDERS_PATH, encoding="utf-8", dtype={"Postal Code": "string"})
people_raw = pd.read_csv(PEOPLE_PATH, encoding="utf-8")
returns_raw = pd.read_csv(RETURNS_PATH, encoding="utf-8")

datasets_raw = {
    "orders": orders_raw.rename(columns=to_snake_case),
    "people": people_raw.rename(columns=to_snake_case),
    "returns": returns_raw.rename(columns=to_snake_case),
}

for name, df in datasets_raw.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns")
    print(df.columns.tolist())
    print()

## 3. Exploration de la qualite des donnees

On inspecte les types, les valeurs manquantes et les doublons avant de lancer le nettoyage.

In [ ]:
quality_summary = pd.concat(
    [build_quality_summary(name, df) for name, df in datasets_raw.items()],
    ignore_index=True,
)
print(quality_summary.to_string(index=False))

for name, df in datasets_raw.items():
    print(f"\n{name.upper()} dtypes")
    print(df.dtypes.to_string())

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    print(f"\n{name.upper()} missing values")
    if missing.empty:
        print("No missing values")
    else:
        print(missing.to_string())

    print(f"\n{name.upper()} duplicate rows: {int(df.duplicated().sum())}")

## 4. Nettoyage des jeux de donnees

Cette etape standardise les textes, corrige les types, conserve les valeurs nulles utiles et supprime uniquement les doublons exacts.

In [ ]:
def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_text_columns(df)
    df = df.drop_duplicates().copy()

    df["row_id"] = pd.to_numeric(df["row_id"], errors="coerce").astype("Int64")
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    df["ship_date"] = pd.to_datetime(df["ship_date"], errors="coerce")
    df["postal_code"] = clean_postal_code(df["postal_code"])

    numeric_columns = [
        "sales",
        "quantity",
        "discount",
        "profit",
        "shipping_cost",
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df["quantity"] = df["quantity"].astype("Int64")
    return df


def clean_people(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_text_columns(df)
    df = df.drop_duplicates().copy()
    df = df.rename(columns={"person": "manager_name"})
    return df


def clean_returns(df: pd.DataFrame) -> pd.DataFrame:
    df = normalize_text_columns(df)
    df = df.drop_duplicates().copy()
    df["returned"] = df["returned"].str.title()
    df["is_returned"] = df["returned"].eq("Yes")
    return df[["order_id", "is_returned"]]


orders_clean = clean_orders(datasets_raw["orders"])
people_clean = clean_people(datasets_raw["people"])
returns_clean = clean_returns(datasets_raw["returns"])

print("orders_clean shape:", orders_clean.shape)
print("people_clean shape:", people_clean.shape)
print("returns_clean shape:", returns_clean.shape)

## 5. Fusion des jeux de donnees

Le pipeline applique des jointures `left` a partir de `orders` afin de conserver toutes les lignes de vente, meme si une table de correspondance ne matche pas.

In [ ]:
sales_enriched = (
    orders_clean
    .merge(returns_clean, on="order_id", how="left")
    .merge(people_clean, on="region", how="left")
)

sales_enriched["is_returned"] = sales_enriched["is_returned"].astype("boolean").fillna(False)

print("orders rows :", len(orders_clean))
print("merged rows :", len(sales_enriched))
print("rows with manager_name null:", int(sales_enriched["manager_name"].isna().sum()))

## 6. Creation des indicateurs metier

Cette etape ajoute des colonnes analytiques utiles pour le reporting et les futures requetes SQL.

In [ ]:
sales_enriched["sales_amount"] = sales_enriched["sales"]
sales_enriched["unit_price"] = sales_enriched["sales"].div(sales_enriched["quantity"])
sales_enriched["shipping_delay_days"] = (sales_enriched["ship_date"] - sales_enriched["order_date"]).dt.days
sales_enriched["profit_margin"] = sales_enriched["profit"].div(sales_enriched["sales"].where(sales_enriched["sales"].ne(0)))

priority_columns = [
    "row_id",
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "order_priority",
    "customer_id",
    "customer_name",
    "segment",
    "product_id",
    "product_name",
    "category",
    "sub_category",
    "country",
    "state",
    "city",
    "postal_code",
    "region",
    "market",
    "manager_name",
    "sales",
    "sales_amount",
    "quantity",
    "unit_price",
    "discount",
    "profit",
    "profit_margin",
    "shipping_cost",
    "shipping_delay_days",
    "is_returned",
]

remaining_columns = [column for column in sales_enriched.columns if column not in priority_columns]
final_dataset = sales_enriched[priority_columns + remaining_columns].copy()

## 7. Validation finale

Ces controles permettent de verifier que le dataset nettoye reste coherent avant l'export.

In [ ]:
assert people_clean["region"].is_unique, "people.region must be unique"
assert returns_clean["order_id"].is_unique, "returns.order_id must be unique"
assert len(final_dataset) == len(orders_clean), "Left joins must keep all order rows"
assert final_dataset["order_date"].notna().all(), "order_date contains invalid values"
assert final_dataset["ship_date"].notna().all(), "ship_date contains invalid values"
assert final_dataset["ship_date"].ge(final_dataset["order_date"]).all(), "ship_date must be after order_date"
assert final_dataset["quantity"].gt(0).all(), "quantity must be strictly positive"
assert final_dataset["sales"].ge(0).all(), "sales must be non-negative"
assert final_dataset["discount"].between(0, 1).all(), "discount must stay between 0 and 1"
assert final_dataset["unit_price"].notna().all(), "unit_price should be available for all rows"

validation_summary = pd.DataFrame([
    {"check": "orders_kept_after_left_joins", "result": len(final_dataset) == len(orders_clean)},
    {"check": "people_region_unique", "result": people_clean["region"].is_unique},
    {"check": "returns_order_id_unique", "result": returns_clean["order_id"].is_unique},
    {"check": "valid_order_dates", "result": final_dataset["order_date"].notna().all()},
    {"check": "valid_ship_dates", "result": final_dataset["ship_date"].notna().all()},
    {"check": "ship_after_order", "result": final_dataset["ship_date"].ge(final_dataset["order_date"]).all()},
    {"check": "quantity_positive", "result": final_dataset["quantity"].gt(0).all()},
    {"check": "sales_non_negative", "result": final_dataset["sales"].ge(0).all()},
    {"check": "discount_between_0_and_1", "result": final_dataset["discount"].between(0, 1).all()},
])

print(validation_summary.to_string(index=False))

## 8. Export du CSV final

Les dates sont converties au format texte ISO afin de faciliter l'import dans PostgreSQL.

In [ ]:
export_dataset = final_dataset.copy()
export_dataset["order_date"] = export_dataset["order_date"].dt.strftime("%Y-%m-%d")
export_dataset["ship_date"] = export_dataset["ship_date"].dt.strftime("%Y-%m-%d")

export_dataset.to_csv(OUTPUT_PATH, index=False)
print(f"File exported to: {OUTPUT_PATH}")
print(f"Final shape: {export_dataset.shape}")

reloaded = pd.read_csv(
    OUTPUT_PATH,
    dtype={"postal_code": "string"},
    parse_dates=["order_date", "ship_date"],
    true_values=["True"],
    false_values=["False"],
)

assert reloaded.shape == export_dataset.shape, "Reloaded CSV shape mismatch"
assert reloaded.columns.tolist() == export_dataset.columns.tolist(), "Reloaded CSV columns mismatch"

print("CSV reload validation passed.")
print(reloaded.head(5).to_string(index=False))